In [5]:
# Célula 1: Setup e Busca do Melhor Modelo no MLflow
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

# Conectando ao nosso servidor local do MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
experiment_name = "Olist_Recommendation_Baselines"

print(f"🔍 Buscando o melhor baseline no experimento: {experiment_name}")

# Puxando o ID do experimento
experiment = mlflow.get_experiment_by_name(experiment_name)

# Usando a API do MLflow para buscar a Run com o maior NDCG@10
best_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.NDCG_at_10 DESC"],
    max_results=1
).iloc[0]

run_id = best_run.run_id
run_name = best_run['tags.mlflow.runName']
best_ndcg = best_run['metrics.NDCG_at_10']

print(f"✅ Vencedor Encontrado: {run_name}")
print(f"📊 Pontuação (NDCG@10): {best_ndcg:.4f}")
print(f"🆔 Run ID: {run_id}")
print("-" * 50)

# Célula 2: Baixando o artefato, enriquecendo com a categoria e exibindo
client = MlflowClient()

print("⬇️ Baixando as recomendações geradas pelo baseline de produção...")
local_path = client.download_artifacts(run_id, "temporary_baseline_recommendations.csv")
recs_df = pd.read_csv(local_path)

print("📦 Carregando a base de produtos para buscar os nomes (categorias)...")
# Carregando o dataset de produtos direto da pasta raw
products_df = pd.read_csv("../data/raw/olist_products_dataset.csv")

# Fazendo o "INNER JOIN" para o baseline de Popularidade
pop_merged = recs_df[['popularity_baseline']].merge(
    products_df[['product_id', 'product_category_name']], 
    left_on='popularity_baseline', 
    right_on='product_id', 
    how='left'
).rename(columns={'product_category_name': 'popularity_category'})

# Fazendo o "INNER JOIN" para o baseline de Maior Nota
rated_merged = recs_df[['top_rated_baseline']].merge(
    products_df[['product_id', 'product_category_name']], 
    left_on='top_rated_baseline', 
    right_on='product_id', 
    how='left'
).rename(columns={'product_category_name': 'top_rated_category'})

# Juntando tudo num dataframe final bonitão
final_df = pd.DataFrame({
    'Popularity_ID': pop_merged['product_id'],
    'Popularity_Category': pop_merged['popularity_category'],
    'TopRated_ID': rated_merged['product_id'],
    'TopRated_Category': rated_merged['top_rated_category']
})

print("🛍️ Top 5 Recomendações Enriquecidas:")
display(final_df.head(5))

🔍 Buscando o melhor baseline no experimento: Olist_Recommendation_Baselines
✅ Vencedor Encontrado: Run_2_High_TopN
📊 Pontuação (NDCG@10): 0.0896
🆔 Run ID: 7d374a0edaf74ca082197da668f53b6b
--------------------------------------------------
⬇️ Baixando as recomendações geradas pelo baseline de produção...


📦 Carregando a base de produtos para buscar os nomes (categorias)...
🛍️ Top 5 Recomendações Enriquecidas:


,Popularity_ID,Popularity_Category,TopRated_ID,TopRated_Category
0,99a4788cb24856965c36a24e339b6058,cama_mesa_banho,fffdb2d0ec8d6a61f0a0a0db3f25b441,informatica_acessorios
1,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,83aae8023b8feda53259f63e0ec06390,moveis_decoracao
2,422879e10f46682990de24d770e7f83d,ferramentas_jardim,7be620cd314ce6460033ba9017be7a7f,brinquedos
3,d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,7c1e2b3fa0233e46fb3bcdcb9919a72f,papelaria
4,389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,7c6fbb3a5346dfd607386155c6f628f8,beleza_saude
